# 08 – NLP Analysis

### Purpose of the Notebook
Analyse textbasierter Vergabefelder mittels NLP.

### Steps
- TF‑IDF preprocessing
- SVD dimensionality reduction
- NMF topic modelling
- SVM text‑risk classifier (3 classes)
- Export TEXT_RISK_SCORE + NLP features
- Integration into modelling pipeline

--------------------
#### Imports & Setup & Dataset
-------------------

In [1]:
# ---------------------------------------------------------
# Import moduls
# ---------------------------------------------------------


# import standard modules
import pandas as pd
import numpy as np
from pathlib import Path

import sys
from pathlib import Path

In [2]:
# shut off some annoying warnings
import warnings

warnings.filterwarnings("ignore", message="A value is trying to be set on a copy")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
# ---------------------------------------------------------
# Setup style
# ---------------------------------------------------------

# show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# visualisation settings
pd.set_option('display.float_format', '{:,.2f}'.format)

In [4]:
# ---------------------------------------------------------
# Load scripts
# ----------------------------------------------------------

%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# found min directory
PROJECT_ROOT = Path("..").resolve()

# maindirectory sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# import function from script
from my_scripts.nlp_processing import (preprocess_text, build_tfidf_svd, build_nmf_topics,
                               build_text_risk_classifier, predict_text_risk
)
from my_scripts.eda import overview

In [ ]:
# ---------------------------------------------------------
# Import dataset
# ---------------------------------------------------------

df = pd.read_pickle("../data/dataset_comp.pkl")

print("EU dataset:", df.shape)

EU dataset: (4039906, 41)
DE dataset: (303349, 41)


----------------
### NLP PROCESSING

----------

In [6]:
# ---------------------------------------------------------
# Preprocess text 
# ---------------------------------------------------------

text_cols = ["TITLE", "CRIT_CRITERIA", "CRIT_WEIGHTS"]
df = preprocess_text(df, text_cols)


In [9]:
df["TEXT_ALL"] = df["TEXT_ALL"].fillna("").astype(str)

In [17]:
# ---------------------------------------------------------
# TF‑IDF + SVD features
# ---------------------------------------------------------

df_svd, X_tfidf, tfidf_vectorizer, svd_model = build_tfidf_svd(df["TEXT_ALL"])
df = pd.concat([df, df_svd], axis=1)


In [18]:
# ---------------------------------------------------------
# Topic modelling (NMF)
# ---------------------------------------------------------
df_topics, nmf_model = build_nmf_topics(X_tfidf)
df = pd.concat([df, df_topics], axis=1)


In [25]:
df.shape

(4362592, 207)

In [27]:
# ---------------------------------------------------------
# Prepare labels for TEXT_RISK_SCORE
# ---------------------------------------------------------

risk_map = {
    "failed": "high",
    "low": "medium",
    "medium": "medium",
    "high": "low"
}

df["TEXT_RISK_LABEL"] = df["OFFERS_BIN"].map(risk_map)


In [ ]:
# ---------------------------------------------------------
# Train SVM classifier
# ---------------------------------------------------------

df_text = df[df["OFFERS_BIN"].notna()]

texts = df_text["TEXT_ALL"].fillna("").astype(str)
labels = df_text["OFFERS_BIN"].astype(str)

svm_model, tfidf_svm, label_encoder = build_text_risk_classifier(texts, labels)


In [ ]:
# ---------------------------------------------------------
# Predict TEXT_RISK_SCORE
# ---------------------------------------------------------

df["TEXT_RISK_SCORE"] = predict_text_risk(
    df["TEXT_ALL"],
    svm_model,
    tfidf_svm,
    label_encoder
)



In [ ]:
# drop unnecessary columns
cols_to_drop = [
    "TITLE",
    "CRIT_CRITERIA",
    "CRIT_WEIGHTS",
    "TEXT_ALL"
]

df = df.drop(columns=cols_to_drop, errors="ignore")


#### Notes: NLP Pipeline for Tender Risk Prediction
1. Text fields provide additional signals not captured by structured variables.  
Short titles and evaluation‑related text (TITLE, CRIT_CRITERIA, CRIT_WEIGHTS) reveal complexity, niche requirements, and multi‑criteria scoring patterns that strongly influence bidder participation and failure risk.

2. TF‑IDF offers a scalable and domain‑appropriate representation of tender text.  
It efficiently captures important terms and patterns without requiring heavy linguistic models, making it suitable for millions of records and classical ML workflows.

3. Dimensionality reduction (SVD) converts high‑dimensional TF‑IDF vectors into compact numerical features.  
This reduces sparsity, stabilizes downstream models, and enables seamless integration with structured predictors such as CPV, procedure type, and tender value.

4. Topic modelling (NMF) introduces interpretable thematic structure.  
Extracted topics highlight procurement areas with systematically higher failure rates, improving interpretability and analytical insight.

5. A text‑based classifier (TF‑IDF + SVM) produces a high‑level TEXT_RISK_SCORE.  
It learns patterns associated with failed, medium‑risk, and safe tenders based solely on text, generating a categorical risk signal usable even when raw text is unavailable.

6. TEXT_RISK_SCORE is essential for downstream applications such as the risk simulator.  
It allows the model to incorporate text‑derived risk information without requiring free‑text input, enabling scenario simulations based on a small set of structured parameters.

7. The combined pipeline remains interpretable, scalable, and robust.  
TF‑IDF + SVD ensures numerical stability, NMF adds thematic insight, and SVM provides a practical risk score — together forming a balanced NLP module that strengthens the overall tender risk prediction model.

---------
### SAVE DATASET

--------

In [ ]:
df.to_pickle("../data/dataset_nlp.pkl")
df_de.to_pickle("../data/dataset_de_nlp.pkl")